# SKAB Anomaly Detection — XGBoost & Random Forest on All Datasets
**Datasets:** valve1 (16 files), valve2 (4 files), other (14 files)  
**Algorithms:** XGBoost, Random Forest (supervised)  
**Evaluation:** F1, Precision, Recall, ROC-AUC — per dataset + per file  
**Note:** Runs locally. Data loaded from relative paths alongside this notebook.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn

## Cell 2 — Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, classification_report
)

# Path to SKAB data (relative to this notebook)
BASE_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
print(f'Base directory: {BASE_DIR}')

FEATURE_COLS = [
    'Accelerometer1RMS', 'Accelerometer2RMS', 'Current',
    'Pressure', 'Temperature', 'Thermocouple',
    'Voltage', 'Volume Flow RateRMS'
]

DATASETS = {
    'valve1': [f'valve1/{i}.csv' for i in range(16)],
    'valve2': [f'valve2/{i}.csv' for i in range(4)],
    'other':  [f'other/{f}' for f in sorted(os.listdir(os.path.join(BASE_DIR, 'other'))) if f.endswith('.csv')]
}

print('\nDataset file counts:')
for name, files in DATASETS.items():
    print(f'  {name}: {len(files)} files')

## Cell 3 — Load All Data

In [ ]:
def load_csv(path):
    df = pd.read_csv(path, sep=';', parse_dates=['datetime'])
    return df.sort_values('datetime').reset_index(drop=True)

# Anomaly-free baseline (for scaler fitting)
baseline_df = load_csv(os.path.join(BASE_DIR, 'anomaly-free', 'anomaly-free.csv'))
print(f'Baseline shape: {baseline_df.shape}')

# Load all datasets
all_data = {}
for dataset_name, file_list in DATASETS.items():
    dfs = []
    for fpath in file_list:
        full_path = os.path.join(BASE_DIR, fpath)
        if os.path.exists(full_path):
            df = load_csv(full_path)
            df['source_file'] = os.path.basename(fpath)
            dfs.append(df)
        else:
            print(f'  WARNING: {full_path} not found, skipping.')
    combined = pd.concat(dfs, ignore_index=True)
    all_data[dataset_name] = combined
    print(f'{dataset_name}: {len(dfs)} files, {len(combined)} rows, '
          f'anomaly rate: {combined["anomaly"].mean():.1%}')

## Cell 4 — Feature Engineering

In [ ]:
def add_rolling_features(df, window=5):
    df = df.copy()
    for col in FEATURE_COLS:
        df[f'{col}_roll_mean'] = df[col].rolling(window, min_periods=1).mean()
        df[f'{col}_roll_std']  = df[col].rolling(window, min_periods=1).std().fillna(0)
    return df

EXTENDED_COLS = FEATURE_COLS + \
    [f'{c}_roll_mean' for c in FEATURE_COLS] + \
    [f'{c}_roll_std'  for c in FEATURE_COLS]

# Fit scaler on anomaly-free baseline only
baseline_feat = add_rolling_features(baseline_df)
scaler = StandardScaler()
scaler.fit(baseline_feat[EXTENDED_COLS])

# Apply to all datasets
dataset_features = {}
for name, df in all_data.items():
    feat_df = add_rolling_features(df)
    X = scaler.transform(feat_df[EXTENDED_COLS])
    y = feat_df['anomaly'].values
    groups = feat_df['source_file'].values
    dataset_features[name] = (X, y, groups, feat_df)
    print(f'{name}: X={X.shape}, y={y.shape}, anomaly rate={y.mean():.1%}')

## Cell 5 — Train & Evaluate XGBoost and Random Forest (Per Dataset)

In [ ]:
def evaluate_models(X, y, groups, dataset_name, n_splits=5):
    neg = (y == 0).sum()
    pos = (y == 1).sum()
    scale_pos = neg / pos if pos > 0 else 1.0

    # Adjust splits if fewer unique groups than n_splits
    n_unique_groups = len(np.unique(groups))
    actual_splits = min(n_splits, n_unique_groups)

    models = {
        'XGBoost': XGBClassifier(
            n_estimators=300, max_depth=6,
            scale_pos_weight=scale_pos,
            eval_metric='logloss',
            random_state=42, n_jobs=-1
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=300,
            class_weight='balanced',
            random_state=42, n_jobs=-1
        )
    }

    gkf = GroupKFold(n_splits=actual_splits)
    results = {}

    print(f'\n{"="*60}')
    print(f'Dataset: {dataset_name}  |  Rows: {len(y)}  |  Folds: {actual_splits}')
    print(f'{"="*60}')

    for model_name, clf in models.items():
        print(f'\n--- {model_name} ---')
        all_preds  = np.zeros(len(y))
        all_probas = np.zeros(len(y))

        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
            clf.fit(X[train_idx], y[train_idx])
            all_preds[test_idx]  = clf.predict(X[test_idx])
            all_probas[test_idx] = clf.predict_proba(X[test_idx])[:, 1]
            fold_f1 = f1_score(y[test_idx], all_preds[test_idx], zero_division=0)
            print(f'  Fold {fold+1}  F1: {fold_f1:.4f}')

        print(classification_report(y, all_preds, target_names=['Normal', 'Anomaly'], zero_division=0))
        print(f'ROC-AUC: {roc_auc_score(y, all_probas):.4f}')

        results[model_name] = {
            'preds': all_preds,
            'probas': all_probas,
            'f1':        round(f1_score(y, all_preds, zero_division=0), 4),
            'precision': round(precision_score(y, all_preds, zero_division=0), 4),
            'recall':    round(recall_score(y, all_preds, zero_division=0), 4),
            'roc_auc':   round(roc_auc_score(y, all_probas), 4)
        }

    return results


all_results = {}
for dataset_name, (X, y, groups, feat_df) in dataset_features.items():
    all_results[dataset_name] = evaluate_models(X, y, groups, dataset_name)

## Cell 6 — Summary Table: All Datasets × Both Models

In [ ]:
rows = []
for dataset_name, model_results in all_results.items():
    for model_name, metrics in model_results.items():
        rows.append({
            'Dataset':   dataset_name,
            'Model':     model_name,
            'F1':        metrics['f1'],
            'Precision': metrics['precision'],
            'Recall':    metrics['recall'],
            'ROC-AUC':   metrics['roc_auc']
        })

summary_df = pd.DataFrame(rows)
print('\n========== FULL SUMMARY ==========\n')
print(summary_df.to_string(index=False))

## Cell 7 — Heatmap: F1 Score by Dataset and Model

In [ ]:
pivot_f1 = summary_df.pivot(index='Dataset', columns='Model', values='F1')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# F1 heatmap
sns.heatmap(pivot_f1, annot=True, fmt='.4f', cmap='YlGn',
            vmin=0, vmax=1, ax=axes[0], linewidths=0.5)
axes[0].set_title('F1 Score by Dataset & Model', fontweight='bold')

# ROC-AUC heatmap
pivot_auc = summary_df.pivot(index='Dataset', columns='Model', values='ROC-AUC')
sns.heatmap(pivot_auc, annot=True, fmt='.4f', cmap='Blues',
            vmin=0, vmax=1, ax=axes[1], linewidths=0.5)
axes[1].set_title('ROC-AUC by Dataset & Model', fontweight='bold')

plt.tight_layout()
plt.savefig('heatmap_all_datasets.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: heatmap_all_datasets.png')

## Cell 8 — Per-File F1 Breakdown (All Datasets)

In [ ]:
def per_file_f1(dataset_name, model_name, feat_df, y, preds):
    rows = []
    for fname in sorted(feat_df['source_file'].unique()):
        mask = feat_df['source_file'].values == fname
        f1  = f1_score(y[mask], preds[mask], zero_division=0)
        anom = y[mask].mean()
        rows.append({'Dataset': dataset_name, 'Model': model_name,
                     'File': fname, 'F1': round(f1, 4), 'Anomaly Rate': round(anom, 3)})
    return rows

per_file_rows = []
for dataset_name, (X, y, groups, feat_df) in dataset_features.items():
    for model_name, metrics in all_results[dataset_name].items():
        per_file_rows.extend(
            per_file_f1(dataset_name, model_name, feat_df, y, metrics['preds'])
        )

per_file_df = pd.DataFrame(per_file_rows)
print('\n========== PER-FILE F1 BREAKDOWN ==========\n')
print(per_file_df.to_string(index=False))

## Cell 9 — Bar Charts: Per-File F1 for Each Dataset

In [ ]:
dataset_names = list(DATASETS.keys())
fig, axes = plt.subplots(len(dataset_names), 1,
                         figsize=(14, 5 * len(dataset_names)),
                         constrained_layout=True)

colors = {'XGBoost': 'steelblue', 'Random Forest': 'darkorange'}

for ax, dataset_name in zip(axes, dataset_names):
    subset = per_file_df[per_file_df['Dataset'] == dataset_name]
    files  = sorted(subset['File'].unique())
    x = np.arange(len(files))
    width = 0.35

    for i, model_name in enumerate(['XGBoost', 'Random Forest']):
        model_data = subset[subset['Model'] == model_name].set_index('File')
        f1_vals = [model_data.loc[f, 'F1'] if f in model_data.index else 0 for f in files]
        ax.bar(x + i * width, f1_vals, width, label=model_name,
               color=colors[model_name], alpha=0.85)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(files, rotation=45, ha='right')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('F1 Score')
    ax.set_title(f'Per-File F1 Score — {dataset_name}', fontweight='bold')
    ax.legend()
    ax.axhline(0.8, color='red', linestyle='--', lw=1, alpha=0.5, label='0.8 threshold')

plt.savefig('per_file_f1_all_datasets.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_file_f1_all_datasets.png')

## Cell 10 — Best Model Per Dataset

In [ ]:
print('\n========== BEST MODEL PER DATASET (by F1) ==========\n')
best = summary_df.loc[summary_df.groupby('Dataset')['F1'].idxmax()]
print(best.to_string(index=False))

print('\n========== OVERALL AVERAGE (across all datasets) ==========\n')
overall = summary_df.groupby('Model')[['F1', 'Precision', 'Recall', 'ROC-AUC']].mean().round(4)
print(overall)